In [1]:
import numpy as np


In [2]:
class Neuron:
    def __init__(self, num_inputs, activation_function):
        self.weights = np.random.uniform(size=num_inputs)
        self.bias = np.random.uniform()
        self.activation_function = {"0":self.sigmoid, "1":self.tanh}[activation_function]
        self.derivative_function = {"0":self.sigmoid_derivative, "1":self.tanh_derivative}[activation_function]

    def activate(self, inputs):
        self.inputs = inputs
        self.output = self.activation_function(np.dot(inputs, self.weights) + self.bias)
        return self.output

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self):
        return self.output * (1 - self.output)

    def tanh(self, x):
        return 2*self.sigmoid(2*x) - 1
    
    def tanh_derivative(self):
        return 1 - self.output ** 2

    def update_weights(self, delta, learning_rate):
        self.weights += learning_rate * delta * self.inputs
        self.bias += learning_rate * delta


In [3]:
class Layer:
    def __init__(self, num_neurons, num_inputs_per_neuron, activation_function):
        self.neurons = [Neuron(num_inputs_per_neuron, activation_function) for _ in range(num_neurons)]

    def forward(self, inputs):
        return np.array([neuron.activate(inputs) for neuron in self.neurons])

    def backward(self, errors, learning_rate):
        deltas = []
        for i, neuron in enumerate(self.neurons):
            delta = errors[i] * neuron.derivative_function()
            neuron.update_weights(delta, learning_rate)
            deltas.append(delta)
        return np.dot(np.array([neuron.weights for neuron in self.neurons]).T, deltas)


In [4]:
class NeuralNetwork:
    def __init__(self, layers, learning_rate=0.1, epochs=10000, activation_function="0"):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.layers = []

        # Initialize layers
        for i in range(len(layers) - 1):
            self.layers.append(Layer(layers[i+1], layers[i], activation_function))

    def train(self, inputs, outputs):
        for epoch in range(self.epochs):
            total_error = 0
            for x, y in zip(inputs, outputs):
                # Forward pass
                activations = [x]
                for layer in self.layers:
                    activations.append(layer.forward(activations[-1]))

                # Calculate error
                output_errors = y - activations[-1]
                total_error += np.sum(output_errors ** 2)

                # Backward pass
                errors = output_errors
                for i in reversed(range(len(self.layers))):
                    errors = self.layers[i].backward(errors, self.learning_rate)

            # Print MSE every 1000 epochs
            # if epoch % 1000 == 0:
            #     mse = total_error / len(inputs)
            #     print(f'Epoch {epoch}, MSE: {mse}')

    def predict(self, inputs):
        activations = inputs
        for layer in self.layers:
            activations = layer.forward(activations)
        return activations


In [5]:
inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
outputs = {
    "AND": np.array([[0], [0], [0], [1]]),
    "OR" : np.array([[0], [1], [1], [1]]),
    "XOR": np.array([[0], [1], [1], [0]])
    }


In [ ]:
def inputReader():
    modes = {
        "AND": 1,
        "OR" : 2,
        "XOR": 4,
        "ALL": 7
    }
    
    mode = input()
    mode = modes[mode]
    func = input()
    hidden_layers = int(input())
    neurons_per_layer = int(input())
    layers = [2] + hidden_layers*[neurons_per_layer] + [1]

    return mode, func, layers

def trainNeuralNetwork(mode, func, layers):
    nn = NeuralNetwork(layers,epochs=10000, learning_rate=0.1, activation_function=func)
    nn.train(inputs, outputs[mode])

    predicted_output = np.array([nn.predict(x) for x in inputs])
    print(mode,"\n(0,0) -> ", predicted_output[0], "\n(0,1) -> ", predicted_output[1], "\n(1,0) -> ", predicted_output[2], "\n(1,1) -> ", predicted_output[3])

mode, func, layers = inputReader()
mode = np.binary_repr(mode, width=3)

if mode[2] == '1':
    trainNeuralNetwork("AND", func, layers)
if mode[1] == '1':
    trainNeuralNetwork("OR" , func, layers)
if mode[0] == '1':
    trainNeuralNetwork("XOR", func, layers)


AND 
(0,0) ->  [0.00566945] 
(0,1) ->  [0.02119884] 
(1,0) ->  [0.0224249] 
(1,1) ->  [0.97193722]
OR 
(0,0) ->  [0.02377504] 
(0,1) ->  [0.98641431] 
(1,0) ->  [0.98621015] 
(1,1) ->  [0.99851257]
XOR 
(0,0) ->  [0.01424229] 
(0,1) ->  [0.96514268] 
(1,0) ->  [0.96632721] 
(1,1) ->  [0.04317813]
